# Research: JEPA pose representation (PyTorch)

# Research lane — JEPA pose representation (PyTorch)

I-JEPA / VICReg over **unlabelled workout video** (research skeleton:
`training/research/jepa_pose_representation.py`). PyTorch here because the
serving pose models and the reference JEPA code are torch-first.

**Decision gate:** does the self-supervised representation beat the supervised
TF baseline for exercise detection / form scoring? Do **not** promote to
serving until the gate passes. The encoder would then be exported to ONNX and
registered as `ModelMetadata name='jepa_pose'`.

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai))         # for the `training.research` namespace package

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['torch', 'torchvision', 'einops'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

from training.research import jepa_pose_representation as jepa
print("JEPA skeleton imported")


In [ ]:
# Video frames -> random context blocks + target blocks (I-JEPA)
# Uses raw workout video; see training/research/jepa_pose_representation.py
print(jepa.__doc__)

In [ ]:
# context encoder (e.g. ResNet-18) + target encoder (EMA) + predictor MLP
import torch, torch.nn as nn
# model = jepa.JepaPose(backbone='resnet18')   # TODO in research lane
print('build context/target encoders + predictor')

In [ ]:
# VICReg/I-JEPA loss over masked latent patches on unlabelled frames
# optimizer.step() with target-encoder momentum update
print('train self-supervised (see decision gate)')

In [ ]:
# Gate check: linear-probe exercise classification + form score correlation
# vs supervised TF baseline. Only proceed to export on pass.
print('compare vs supervised baseline')

In [ ]:
# On gate pass only: torch.onnx.export(encoder) -> jepa_pose_int8.onnx
print('export encoder -> ONNX + register ModelMetadata')